In [1]:
"""
San Diego Weather Helper for Photographers - Initial PostgreSQL Loader
---------------------------------------------------------------------
Creates the PostgreSQL database schema and executes data ingestion for:
- open-meteo-32.72N117.15W12m.csv (Raw API weather file)
- Static rule matrices for photoshoot suitability and client styling metrics

Required packages:
    pip install pandas sqlalchemy psycopg2-binary python-dotenv
"""

'\nSan Diego Weather Helper for Photographers - Initial PostgreSQL Loader\n---------------------------------------------------------------------\nCreates the PostgreSQL database schema and executes data ingestion for:\n- open-meteo-32.72N117.15W12m.csv (Raw API weather file)\n- Static rule matrices for photoshoot suitability and client styling metrics\n\nRequired packages:\n    pip install pandas sqlalchemy psycopg2-binary python-dotenv\n'

In [2]:
from __future__ import annotations

In [3]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.types import Boolean, Date, DateTime, Float, Integer, String

load_dotenv(override=True)

True

In [4]:
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
WEATHER_DATA_CSV = BASE_DIR / "open-meteo-32.72N117.15W12m.csv"

In [5]:
import os

def get_database_url():
    database_url = os.getenv("DATABASE_URL")
    if database_url:
        return database_url.strip()
        
    # The clean, standard direct connection details
    user = "postgres"
    password = "!m1d1000O252525"
    host = "db.bcdnwqqntuqczwvdpdgq.supabase.co"  
    port = "5432"
    db_name = "postgres"

    return f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db_name}"

In [6]:
def table_reset_enabled() -> bool:
    return os.getenv("RESET_TABLES", "true").strip().lower() in {"1", "true", "yes", "y"}

In [7]:
def create_schema(engine) -> None:
    drop_sql = """
    DROP TABLE IF EXISTS public.current_weather CASCADE;
    DROP TABLE IF EXISTS public.daily_forecast CASCADE;
    DROP TABLE IF EXISTS public.weather_code CASCADE;
    DROP TABLE IF EXISTS public.shoot_suitability CASCADE;
    DROP TABLE IF EXISTS public.styling_alert CASCADE;
    """

    create_sql = """
    CREATE TABLE IF NOT EXISTS public.weather_code (
        weather_code_id INTEGER PRIMARY KEY,
        description TEXT NOT NULL);

    CREATE TABLE IF NOT EXISTS public.current_weather (
        current_id SERIAL PRIMARY KEY,
        read_time TIMESTAMP NOT NULL,
        temperature_2m DOUBLE PRECISION NOT NULL,
        relative_humidity_2m INTEGER NOT NULL,
        apparent_temperature DOUBLE PRECISION NOT NULL,
        precipitation DOUBLE PRECISION NOT NULL,
        weather_code_id INTEGER NOT NULL REFERENCES public.weather_code(weather_code_id),
        cloud_cover INTEGER NOT NULL,
        wind_speed_10m DOUBLE PRECISION NOT NULL);

    CREATE TABLE IF NOT EXISTS public.daily_forecast (
        forecast_id BIGSERIAL PRIMARY KEY,
        forecast_date DATE NOT NULL UNIQUE,
        weather_code_id INTEGER NOT NULL REFERENCES public.weather_code(weather_code_id),
        temperature_2m_max DOUBLE PRECISION NOT NULL,
        temperature_2m_min DOUBLE PRECISION NOT NULL,
        sunrise TIMESTAMP NOT NULL,
        sunset TIMESTAMP NOT NULL,
        precipitation_sum DOUBLE PRECISION NOT NULL,
        wind_speed_10m_max DOUBLE PRECISION NOT NULL);

    CREATE TABLE IF NOT EXISTS public.shoot_suitability (
        suitability_id INTEGER PRIMARY KEY,
        shoot_type_name VARCHAR(50) UNIQUE NOT NULL,
        max_wind_tolerance DOUBLE PRECISION NOT NULL,
        max_precipitation_tolerance DOUBLE PRECISION NOT NULL,
        temperature_floor DOUBLE PRECISION NOT NULL,
        temperature_ceiling DOUBLE PRECISION NOT NULL);

    CREATE TABLE IF NOT EXISTS public.styling_alert (
        alert_id INTEGER PRIMARY KEY,
        metric_target VARCHAR(30) NOT NULL,
        threshold_min DOUBLE PRECISION NOT NULL,
        threshold_max DOUBLE PRECISION NOT NULL,
        alert_text TEXT NOT NULL,
        styling_recommendation TEXT NOT NULL);
    """

    with engine.connect() as conn:
        if table_reset_enabled():
            conn.execute(text(drop_sql))
            print("Existing tables dropped to execute a clean schema load sequence.")
        
        conn.execute(text(create_sql))
        conn.commit()
    print("Relational database schema structures generated cleanly.")

In [8]:
def transform_weather_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not WEATHER_DATA_CSV.exists():
        raise FileNotFoundError(f"Missing target file dependency: {WEATHER_DATA_CSV}")

    weather_code_records = [
        {"weather_code_id": 0, "description": "Clear sky"},
        {"weather_code_id": 1, "description": "Mainly clear"},
        {"weather_code_id": 2, "description": "Partly cloudy"},
        {"weather_code_id": 3, "description": "Overcast"},
        {"weather_code_id": 45, "description": "Fog and depositing rime fog"}]
    weather_code_df = pd.DataFrame(weather_code_records)

    current_df = pd.read_csv(WEATHER_DATA_CSV, skiprows=3, nrows=1)
    current_df = current_df.rename(
        columns={
            "time": "read_time",
            "temperature_2m (°C)": "temperature_2m",
            "relative_humidity_2m (%)": "relative_humidity_2m",
            "apparent_temperature (°C)": "apparent_temperature",
            "precipitation (mm)": "precipitation",
            "weather_code (wmo code)": "weather_code_id",
            "cloud_cover (%)": "cloud_cover",
            "wind_speed_10m (km/h)": "wind_speed_10m"})

    current_df["temperature_2m"] = (current_df["temperature_2m"] * 9/5) + 32
    current_df["apparent_temperature"] = (current_df["apparent_temperature"] * 9/5) + 32
    current_df["precipitation"] = current_df["precipitation"] / 25.4
    current_df["wind_speed_10m"] = current_df["wind_speed_10m"] / 1.60934
    current_df["read_time"] = pd.to_datetime(current_df["read_time"])

    daily_df = pd.read_csv(WEATHER_DATA_CSV, skiprows=6)
    daily_df = daily_df.rename(
        columns={
            "time": "forecast_date",
            "weather_code (wmo code)": "weather_code_id",
            "temperature_2m_max (°C)": "temperature_2m_max",
            "temperature_2m_min (°C)": "temperature_2m_min",
            "sunrise (iso8601)": "sunrise",
            "sunset (iso8601)": "sunset",
            "precipitation_sum (mm)": "precipitation_sum",
            "wind_speed_10m_max (km/h)": "wind_speed_10m_max"})

    daily_df["temperature_2m_max"] = (daily_df["temperature_2m_max"] * 9/5) + 32
    daily_df["temperature_2m_min"] = (daily_df["temperature_2m_min"] * 9/5) + 32
    daily_df["precipitation_sum"] = daily_df["precipitation_sum"] / 25.4
    daily_df["wind_speed_10m_max"] = daily_df["wind_speed_10m_max"] / 1.60934
    
    daily_df["forecast_date"] = pd.to_datetime(daily_df["forecast_date"]).dt.date
    daily_df["sunrise"] = pd.to_datetime(daily_df["sunrise"])
    daily_df["sunset"] = pd.to_datetime(daily_df["sunset"])

    print("Source data file parsed, isolated, and transformed into imperial metrics.")
    return weather_code_df, current_df, daily_df

In [9]:
def build_business_rules_tables() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Generates the static data matrices for business intelligence mapping."""
    suitability_records = [
        {"suitability_id": 1, "shoot_type_name": "Beach Portrait", "max_wind_tolerance": 12.0, "max_precipitation_tolerance": 0.0, "temperature_floor": 60.0, "temperature_ceiling": 85.0},
        {"suitability_id": 2, "shoot_type_name": "Indoor Studio", "max_wind_tolerance": 100.0, "max_precipitation_tolerance": 100.0, "temperature_floor": 0.0, "temperature_ceiling": 100.0}
    ]
    shoot_suitability_df = pd.DataFrame(suitability_records)

    styling_records = [
        {"alert_id": 1, "metric_target": "humidity", "threshold_min": 75.0, "threshold_max": 100.0, "alert_text": "High frizz risk.", "styling_recommendation": "Advise water-resistant makeup and updo hairstyles."},
        {"alert_id": 2, "metric_target": "wind", "threshold_min": 15.0, "threshold_max": 100.0, "alert_text": "High wind warning.", "styling_recommendation": "Secure loose hair; avoid long flowing veils for outdoor portraiture."}
    ]
    styling_alert_df = pd.DataFrame(styling_records)

    return shoot_suitability_df, styling_alert_df

In [10]:
def write_table(df: pd.DataFrame, table_name: str, engine, dtype: dict) -> None:
    """Automates clean batch-appends into PostgreSQL tables using SQLAlchemy."""
    print(f"Loading data records into the {table_name} table...")
    df.to_sql(
        table_name,
        engine,
        schema="public",
        if_exists="append",
        index=False,
        method="multi",
        chunksize=1000,
        dtype=dtype,)

In [11]:
env_content = """
DB_USER=postgres.bcdnwqqntuqczwvdpdgq
DB_HOST=aws-0-us-east-1.pooler.supabase.com
DB_PORT=6543
DB_NAME=postgres
DB_PASSWORD=!m1d1000O252525
RESET_TABLES=true
"""

with open(".env", "w") as f:
    f.write(env_content.strip())

print("Successfully configured .env for the Supabase Connection Pooler!")

Successfully configured .env for the Supabase Connection Pooler!


In [12]:
import traceback
from sqlalchemy import create_engine

def main() -> None:
    try:
        print("Initializing database connection engine...")
        
        engine = create_engine(get_database_url())
       
        print("Executing data transformations...")
        weather_code_df, current_df, daily_df = transform_weather_data()
        
        print("Building static business rules tables...")
        shoot_suitability_df, styling_alert_df = build_business_rules_tables()
        
        print("Recreating database schema on Supabase Cloud...")
        create_schema(engine)

        print("Writing data batches to data warehouse...")
        write_table(weather_code_df, "weather_code", engine, {"weather_code_id": Integer(), "description": String()})
        write_table(shoot_suitability_df, "shoot_suitability", engine, {"suitability_id": Integer(), "shoot_type_name": String(), "max_wind_tolerance": Float(), "max_precipitation_tolerance": Float(), "temperature_floor": Float(), "temperature_ceiling": Float()})
        write_table(styling_alert_df, "styling_alert", engine, {"alert_id": Integer(), "metric_target": String(), "threshold_min": Float(), "threshold_max": Float(), "alert_text": String(), "styling_recommendation": String()})
        write_table(current_df, "current_weather", engine, {"current_id": Integer(), "read_time": DateTime(), "temperature_2m": Float(), "relative_humidity_2m": Integer(), "apparent_temperature": Float(), "precipitation": Float(), "weather_code_id": Integer(), "cloud_cover": Integer(), "wind_speed_10m": Float()})
        write_table(daily_df, "daily_forecast", engine, {"forecast_id": Integer(), "forecast_date": Date(), "weather_code_id": Integer(), "temperature_2m_max": Float(), "temperature_2m_min": Float(), "sunrise": DateTime(), "sunset": DateTime(), "precipitation_sum": Float(), "wind_speed_10m_max": Float()})

        print("\n" + "=" * 35)
        print("PHOTOGRAPHY ETL DATABASE LOAD COMPLETE")
        print("=" * 35)
        
    except Exception as e:
        print("\n❌ THE PIPELINE CRASHED! CRITICAL ERROR LOGGED BELOW:")
        print("-" * 50)
        traceback.print_exc()
        print("-" * 50)

main()

Initializing database connection engine...
Executing data transformations...
Source data file parsed, isolated, and transformed into imperial metrics.
Building static business rules tables...
Recreating database schema on Supabase Cloud...
Existing tables dropped to execute a clean schema load sequence.
Relational database schema structures generated cleanly.
Writing data batches to data warehouse...
Loading data records into the weather_code table...
Loading data records into the shoot_suitability table...
Loading data records into the styling_alert table...
Loading data records into the current_weather table...
Loading data records into the daily_forecast table...

PHOTOGRAPHY ETL DATABASE LOAD COMPLETE
